### Basic RNN Deep Learning Model steps
1. Gather data
2. Preprocess & cleaning data
3. Feature engineering
4. Build a model
5. Save model into .h5 file
6. Load model from .h5 file

#### Import libraries

In [40]:
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.utils import pad_sequences
from tensorflow.keras.layers import Dense, Input, Embedding, SimpleRNN
from tensorflow.keras.models import Sequential
from sklearn.model_selection import train_test_split
import pickle

In [2]:
train_data = pd.read_csv('twitter_training.csv')

In [3]:
train_data.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [4]:
train_data.shape

(74681, 4)

In [5]:
# We use only 15000 samples to train the model to save time.
data = train_data.sample(15000, random_state=42)

In [6]:
data.shape

(15000, 4)

In [7]:
data

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
34877,6792,Fortnite,Irrelevant,went to go in george's room to find his door w...
21704,4115,CS-GO,Positive,Yo this looks LIT! Team:GO/Overwatch combo
47008,5665,HomeDepot,Negative,Pay attention executive administrators. While ...
7969,9369,Overwatch,Irrelevant,Guy looked at me and says my name was put on t...
454,2476,Borderlands,Positive,one
...,...,...,...,...
7757,9333,Overwatch,Positive,i can play overwatch again (bad)
3887,1872,CallOfDutyBlackopsColdWar,Negative,The base weapons of black ops Cold War are so ...
22629,4275,CS-GO,Neutral,Don'don t just jump me... @CSGO... is medal. t...
66836,7044,johnson&johnson,Irrelevant,for FUCKS SAKE!!!!!!!! FIRST MONTH FOR ONE MON...


### Data exploration

In [8]:
# Check null values
data.isnull().sum()

2401                                                       0
Borderlands                                                0
Positive                                                   0
im getting on borderlands and i will murder you all ,    151
dtype: int64

In [9]:
# Check duplicate values
data.duplicated().sum()

np.int64(127)

In [10]:
# Check positive column different values
data['Positive'].value_counts()

Positive
Negative      4488
Positive      4275
Neutral       3568
Irrelevant    2669
Name: count, dtype: int64

### Drop unwanted columns and cleaning data

In [11]:
data.drop(columns=['2401', 'Borderlands'], inplace=True)

In [12]:
data

,Positive,"im getting on borderlands and i will murder you all ,"
34877,Irrelevant,went to go in george's room to find his door w...
21704,Positive,Yo this looks LIT! Team:GO/Overwatch combo
47008,Negative,Pay attention executive administrators. While ...
7969,Irrelevant,Guy looked at me and says my name was put on t...
454,Positive,one
...,...,...
7757,Positive,i can play overwatch again (bad)
3887,Negative,The base weapons of black ops Cold War are so ...
22629,Neutral,Don'don t just jump me... @CSGO... is medal. t...
66836,Irrelevant,for FUCKS SAKE!!!!!!!! FIRST MONTH FOR ONE MON...


In [13]:
# Remove null values
data.dropna(inplace=True)

In [14]:
data.isnull().sum()

Positive                                                 0
im getting on borderlands and i will murder you all ,    0
dtype: int64

In [15]:
# Remove duplicate values
data.drop_duplicates(inplace=True)

In [16]:
data.duplicated().sum()

np.int64(0)

In [17]:
# Change column names
data.rename(columns={'Positive': 'Sentiment', 'im getting on borderlands and i will murder you all ,': 'Tweets'}, inplace=True)

In [18]:
data.columns

Index(['Sentiment', 'Tweets'], dtype='str')

In [19]:
# Check final data shape
data.shape

(14445, 2)

In [20]:
# Remove Irrelevant sentiment from data
data = data[data['Sentiment'].isin(['Neutral', 'Positive', 'Negative'])]

In [21]:
data['Sentiment'].value_counts()

Sentiment
Negative    4342
Positive    4101
Neutral     3424
Name: count, dtype: int64

### Feature engineering & encoding

In [22]:
label = LabelEncoder()

In [23]:
# Apply label encoding to Sentiment column
data['Sentiment'] = label.fit_transform(data['Sentiment'])

In [24]:
data

,Sentiment,Tweets
21704,2,Yo this looks LIT! Team:GO/Overwatch combo
47008,0,Pay attention executive administrators. While ...
454,2,one
58076,0,@Rainbow6Game Server are Available in Xbox 🥺
40659,2,so Awesome siege attempt... strangest bleeding...
...,...,...
57537,2,This is not my nice way of saying what GN with...
7757,2,i can play overwatch again (bad)
3887,0,The base weapons of black ops Cold War are so ...
22629,1,Don'don t just jump me... @CSGO... is medal. t...


In [25]:
tokenizer = Tokenizer(oov_token='Noting')

In [26]:
# Apply tokenizer to Tweets column
tokenizer.fit_on_texts(data['Tweets'])

In [27]:
# Convert texts to sequences
sequences = tokenizer.texts_to_sequences(data['Tweets'])
sequences

[[590, 11, 199, 1676, 214, 95, 196, 2523],
 [308,
  2159,
  4021,
  4022,
  231,
  36,
  1001,
  29,
  791,
  898,
  1324,
  10,
  6516,
  7,
  6517,
  4,
  6518,
  145,
  15,
  247,
  193,
  436,
  9,
  1075,
  2,
  3481,
  7,
  2160,
  3095],
 [51],
 [153, 504, 29, 1002, 9, 99, 2321],
 [20,
  250,
  810,
  3096,
  4868,
  4023,
  33,
  2524,
  6519,
  152,
  115,
  13,
  149,
  10123,
  1677,
  243,
  101,
  2,
  2524,
  1516,
  244,
  28,
  250],
 [56, 54, 286, 466, 4, 2, 18],
 [90,
  10124,
  4024,
  919,
  831,
  118,
  14,
  2,
  190,
  2525,
  10125,
  6520,
  17,
  1771,
  190,
  467,
  90,
  969,
  232,
  4,
  4869,
  373,
  36,
  224,
  4,
  90,
  260,
  14,
  2525,
  2,
  6521],
 [10126, 15, 169, 29, 132, 721, 9, 2, 1882, 7, 6522],
 [113, 2, 87, 441, 111, 351, 368, 53, 83, 1163],
 [3, 58, 76, 196, 1256, 15, 650, 6, 50, 1164, 848, 10, 88, 254],
 [1003, 55, 3, 2790, 196],
 [100,
  467,
  28,
  722,
  47,
  2161,
  170,
  1116,
  2322,
  1517,
  32,
  2526,
  71,
  38,
  125,
 

In [28]:
# Apply padding to sequence of data for consistent input length
sequences = pad_sequences(sequences, padding='pre')

In [29]:
sequences

array([[    0,     0,     0, ...,    95,   196,  2523],
       [    0,     0,     0, ...,     7,  2160,  3095],
       [    0,     0,     0, ...,     0,     0,    51],
       ...,
       [    0,     0,     0, ...,    20,   222,  6161],
       [    0,     0,     0, ...,    43,    72, 17872],
       [    0,     0,     0, ...,   784,  1025,  6485]],
      shape=(11867, 99), dtype=int32)

### Train-Test Split

In [30]:
# Save tweets sequences into "X" and sentiment into "y"
X = sequences
y = data['Sentiment']

In [31]:
X

array([[    0,     0,     0, ...,    95,   196,  2523],
       [    0,     0,     0, ...,     7,  2160,  3095],
       [    0,     0,     0, ...,     0,     0,    51],
       ...,
       [    0,     0,     0, ...,    20,   222,  6161],
       [    0,     0,     0, ...,    43,    72, 17872],
       [    0,     0,     0, ...,   784,  1025,  6485]],
      shape=(11867, 99), dtype=int32)

In [32]:
y

21704    2
47008    0
454      2
58076    0
40659    2
        ..
57537    2
7757     2
3887     0
22629    1
56434    2
Name: Sentiment, Length: 11867, dtype: int64

In [33]:
# Split test-train data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [34]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((9493, 99), (2374, 99), (9493,), (2374,))

### Create a RNN model

In [ ]:
# vocab_size is the total number of unique words in the dataset + 1 (bcz index starts from 0)
vocab_size = len(tokenizer.word_index) + 1


model = Sequential([
    # Input(shape=(99,)),
    Embedding(input_dim=vocab_size, output_dim=128, input_length=99, mask_zero=True),
    SimpleRNN(32),
    Dense(3, activation='softmax') 
])

/home/linux/Desktop/projects/demos/vanilla-rnn/.venv/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
E0000 00:00:1776428144.053387  195996 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [36]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [37]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [43]:
model.fit(X_train, y_train, epochs=100, validation_data=(X_test, y_test),batch_size=64)

Epoch 1/100
149/149 ━━━━━━━━━━━━━━━━━━━━ 10s 65ms/step - accuracy: 0.9890 - loss: 0.0285 - val_accuracy: 0.7047 - val_loss: 1.1491
Epoch 2/100
149/149 ━━━━━━━━━━━━━━━━━━━━ 11s 67ms/step - accuracy: 0.9799 - loss: 0.0539 - val_accuracy: 0.7005 - val_loss: 1.1036
Epoch 3/100
149/149 ━━━━━━━━━━━━━━━━━━━━ 15s 100ms/step - accuracy: 0.9878 - loss: 0.0332 - val_accuracy: 0.7237 - val_loss: 1.0893
Epoch 4/100
149/149 ━━━━━━━━━━━━━━━━━━━━ 15s 102ms/step - accuracy: 0.9920 - loss: 0.0188 - val_accuracy: 0.7174 - val_loss: 1.1018
Epoch 5/100
149/149 ━━━━━━━━━━━━━━━━━━━━ 20s 97ms/step - accuracy: 0.9921 - loss: 0.0165 - val_accuracy: 0.7161 - val_loss: 1.1244
Epoch 6/100
149/149 ━━━━━━━━━━━━━━━━━━━━ 14s 93ms/step - accuracy: 0.9917 - loss: 0.0164 - val_accuracy: 0.7220 - val_loss: 1.1404
Epoch 7/100
149/149 ━━━━━━━━━━━━━━━━━━━━ 14s 91ms/step - accuracy: 0.9919 - loss: 0.0157 - val_accuracy: 0.7178 - val_loss: 1.1753
Epoch 8/100
149/149 ━━━━━━━━━━━━━━━━━━━━ 22s 100ms/step - accuracy: 0.9917 - loss

In [41]:
with open('tokenizer.pickle', 'wb') as file:
    pickle.dump(tokenizer, file)

In [44]:
model.save('sentiment_model.h5')